## Object Shuffle Puzzle Tracker

This notebook tracks 8 PBR 3D objects through a two-stage shuffle puzzle across 96-frame MP4 videos and outputs their final slot positions (0–9).

### 1. Define Constants and Utility Functions

This section defines global constants and helper functions for video frame extraction and encoding.

In [ ]:
import base64
import cv2
import numpy as np
import pandas as pd
import json
import re
import time
import sys
from pathlib import Path
from tqdm.notebook import tqdm

OBJECTS = ["helmet", "avocado", "boombox", "lantern", "camera", "bottle", "duck", "toy_car"]
TOTAL_SLOTS = 10
TOTAL_FRAMES = 96
VIDEO_W, VIDEO_H = 768, 432

SAMPLE_FRAMES = [2, 30, 60, 90]

DATASET_ROOT = "datasets/public"

def extract_frames(video_path: str, frame_indices: list[int]) -> list[np.ndarray]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    for idx in frame_indices:
        clamped = min(idx, total - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, clamped)
        ret, frame = cap.read()
        if ret:
            frames.append((clamped, frame))
    cap.release()
    return frames

def frame_to_b64_jpeg(bgr: np.ndarray, quality: int = 85) -> str:
    ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        raise RuntimeError("JPEG encoding failed")
    return base64.b64encode(buf).decode("utf-8")

# Define approximate slot regions based on video resolution and slot convention
# These are heuristic and might need tuning based on visual inspection
SLOT_CENTERS = [
    (VIDEO_W * 0.1, VIDEO_H * 0.75),  # Slot 0 (Front, Left)
    (VIDEO_W * 0.3, VIDEO_H * 0.75),  # Slot 1
    (VIDEO_W * 0.5, VIDEO_H * 0.75),  # Slot 2
    (VIDEO_W * 0.7, VIDEO_H * 0.75),  # Slot 3
    (VIDEO_W * 0.9, VIDEO_H * 0.75),  # Slot 4 (Front, Right)
    (VIDEO_W * 0.1, VIDEO_H * 0.25),  # Slot 5 (Back, Left)
    (VIDEO_W * 0.3, VIDEO_H * 0.25),  # Slot 6
    (VIDEO_W * 0.5, VIDEO_H * 0.25),  # Slot 7
    (VIDEO_W * 0.7, VIDEO_H * 0.25),  # Slot 8
    (VIDEO_W * 0.9, VIDEO_H * 0.25),  # Slot 9 (Back, Right)
]

# Approximate radius for a slot area (can be tuned)
SLOT_RADIUS_SQ = (VIDEO_W * 0.05)**2 # Use squared distance for efficiency

### 6. Answer Validation & Repair

Function to ensure the answer conforms to the required schema.

### 5. Object Detection with YOLOv8

This section will set up YOLOv8 for detecting the objects in the video frames. We'll use a pre-trained model and then integrate it into our `predict_video` function.

In [21]:
# Install Ultralytics (which includes YOLOv8)
!pip install ultralytics

from ultralytics import YOLO
import torch

# Load a pre-trained YOLOv8 object detection model
# Using 'yolov8n.pt' (nano version) for faster inference
# You might consider 'yolov8m.pt' or 'yolov8l.pt' for better accuracy if performance allows.
model = YOLO('yolov8n.pt')

# You can check the classes the model can detect
# print(model.names)

# If a GPU is available, use it
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Using device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 5.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cpu


### 7. Modified `predict_video` for Object Detection

Now, we'll update the `predict_video` function to use the YOLOv8 model for detecting objects in the sampled frames. This version will print the detections but still return a fallback answer, as full tracking and slot assignment are yet to be implemented.

In [15]:
def repair_answer(answer: dict) -> dict:
    fso = answer.get("final_slot_by_object", {})
    empty = answer.get("empty_slots", []) # Unused in this repair logic, but kept for context
    evidence = answer.get("evidence_frames", {})

    fso = {k.strip().lower(): int(v) for k, v in fso.items()}

    for obj in OBJECTS:
        if obj not in fso:
            fso[obj] = -1

    seen_slots: dict[int, str] = {}
    for obj in OBJECTS:
        s = fso[obj]
        if s in seen_slots:
            fso[obj] = -1
        else:
            if s != -1:
                seen_slots[s] = obj

    occupied = {v for v in fso.values() if v != -1}
    all_slots = set(range(TOTAL_SLOTS))
    free = sorted(list(all_slots - occupied))

    free_iter = iter(free)
    for obj in OBJECTS:
        if fso[obj] == -1:
            try:
                fso[obj] = next(free_iter)
            except StopIteration:
                fso[obj] = 0

    occupied = set(fso.values())
    empty_slots = sorted(list(all_slots - occupied))

    if len(empty_slots) > 2:
        empty_slots = empty_slots[:2]
    elif len(empty_slots) < 2:
        for s in all_slots:
            if s not in occupied and s not in empty_slots:
                empty_slots.append(s)
            if len(empty_slots) == 2:
                break

    return {
        "final_slot_by_object": {obj: fso[obj] for obj in OBJECTS},
        "empty_slots": empty_slots,
        "evidence_frames": evidence,
    }

In [22]:
def predict_video(video_path: str) -> dict:
    frames_with_indices = extract_frames(video_path, SAMPLE_FRAMES)

    if not frames_with_indices:
        print(f"  No frames extracted from {video_path}")
        # Fallback if no frames can be extracted
        fallback = {obj: i for i, obj in enumerate(OBJECTS)}
        return {
            "final_slot_by_object": fallback,
            "empty_slots": [8, 9],
            "evidence_frames": {},
        }

    # Store detections for each frame, focusing on the last frame for final state
    final_frame_detections = []
    last_frame_idx, last_bgr_frame = frames_with_indices[-1] # Get the very last sampled frame

    # print(f"  Detecting objects in final frame ({last_frame_idx}) of {video_path}...")
    rgb_frame = cv2.cvtColor(last_bgr_frame, cv2.COLOR_BGR2RGB)
    results = model(rgb_frame, verbose=False) # verbose=False to suppress output per frame

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()  # bounding box coordinates (x1, y1, x2, y2)
        confidences = r.boxes.conf.cpu().numpy() # confidence scores
        class_ids = r.boxes.cls.cpu().numpy()   # class IDs

        for i in range(len(boxes)):
            class_name = model.names[int(class_ids[i])]
            if class_name in OBJECTS:
                x_center = (boxes[i][0] + boxes[i][2]) / 2
                y_center = (boxes[i][1] + boxes[i][3]) / 2
                final_frame_detections.append({
                    'object_name': class_name,
                    'center': (x_center, y_center),
                    'confidence': confidences[i].item()
                })

    # Initialize final assignments
    final_slot_by_object = {obj: -1 for obj in OBJECTS} # -1 for unassigned
    assigned_slots = set()

    # Assign objects to the nearest unoccupied slot
    for obj_detection in final_frame_detections:
        obj_name = obj_detection['object_name']
        obj_center = obj_detection['center']

        best_slot = -1
        min_distance_sq = float('inf')

        for slot_idx, slot_center in enumerate(SLOT_CENTERS):
            distance_sq = (obj_center[0] - slot_center[0])**2 + \n                          (obj_center[1] - slot_center[1])**2

            # Consider if the detection is reasonably close to a slot center
            if distance_sq < SLOT_RADIUS_SQ and slot_idx not in assigned_slots:
                if distance_sq < min_distance_sq:
                    min_distance_sq = distance_sq
                    best_slot = slot_idx

        if best_slot != -1:
            final_slot_by_object[obj_name] = best_slot
            assigned_slots.add(best_slot)

    # Identify empty slots
    all_slot_indices = set(range(TOTAL_SLOTS))
    empty_slots = sorted(list(all_slot_indices - assigned_slots))

    # Use repair_answer to handle any remaining inconsistencies (e.g., unassigned objects)
    repaired_answer = repair_answer({
        "final_slot_by_object": final_slot_by_object,
        "empty_slots": empty_slots,
        "evidence_frames": {} # Evidence frames not yet implemented in ML/DL approach
    })

    return repaired_answer

### 8. Main Execution Logic

Load the dataset, process each video, and generate the submission file. This section also includes an optional evaluation for training data.

In [20]:
import os

# Configuration
csv_filename = "test.csv"  # Change to "train.csv" to evaluate on the training set
output_path = "/working/submission.csv"
limit_videos = None # Set to an integer to process a subset of videos (e.g., 5)

csv_path = Path(DATASET_ROOT) / csv_filename
if not csv_path.exists():
    sys.exit(f"ERROR: CSV file not found: {csv_path}")
df = pd.read_csv(csv_path)
if limit_videos:
    df = df.head(limit_videos)

# Corrected video directory path
video_dir = Path(DATASET_ROOT)

if not video_dir.exists():
    sys.exit(f"ERROR: Videos directory not found: {video_dir}")

is_train = "answer_json" in df.columns
if is_train:
    print(f"Training CSV detected — will compute accuracy after prediction.")

records = []
correct_objects = 0
total_objects = 0
empty_jaccard_sum = 0.0

# Placeholder for ML/DL predict_video function
def predict_video(video_path: str) -> dict:
    # This function will be implemented with ML/DL logic later
    print(f"  [ML/DL Placeholder] Processing {video_path}")
    fallback = {obj: i for i, obj in enumerate(OBJECTS)}
    return {
        "final_slot_by_object": fallback,
        "empty_slots": [8, 9],
        "evidence_frames": {},
    }

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing videos"):
    # Corrected video file path construction
    video_file = video_dir / row["video"]
    if not video_file.exists():
        print(f"  WARNING: video not found: {video_file} — using fallback")
        answer = {
            "final_slot_by_object": {obj: i for i, obj in enumerate(OBJECTS)},
            "empty_slots": [8, 9],
            "evidence_frames": {},
        }
    else:
        answer = predict_video(str(video_file))

    answer_json_str = json.dumps(answer)
    records.append({"id": row["id"], "answer_json": answer_json_str})

    if is_train and pd.notna(row.get("answer_json")):
        try:
            gt = json.loads(row["answer_json"])
            gt_fso = gt.get("final_slot_by_object", {})
            gt_empty = set(gt.get("empty_slots", []))
            pred_fso = answer["final_slot_by_object"]
            pred_empty = set(answer["empty_slots"])

            for obj in OBJECTS:
                total_objects += 1
                if gt_fso.get(obj) == pred_fso.get(obj):
                    correct_objects += 1

            inter = len(gt_empty & pred_empty)
            union = len(gt_empty | pred_empty)
            empty_jaccard_sum += (inter / union) if union > 0 else 1.0
        except Exception as e:
            print(f"Error during ground truth evaluation for {row['id']}: {e}")

# Create the output directory if it doesn't exist
os.makedirs(Path(output_path).parent, exist_ok=True)
out_path = Path(output_path)
out_df = pd.DataFrame(records, columns=["id", "answer_json"])
out_df.to_csv(out_path, index=False)
print(f"\nSubmission written to: {out_path}  ({len(records)} rows)")

if is_train and total_objects > 0:
    obj_acc = correct_objects / total_objects
    empty_jac = empty_jaccard_sum / len(records)
    row_score = 0.78 * obj_acc + 0.17 * empty_jac
    print(f"\n=== Training-set evaluation ===")
    print(f"  Object slot accuracy : {obj_acc:.4f}  ({correct_objects}/{total_objects})")
    print(f"  Empty slot Jaccard   : {empty_jac:.4f}")
    print(f"  Estimated row score  : {row_score:.4f}")
else:
  print("Evaluation skipped as not running on training data or no objects were processed.")

Processing videos:   0%|          | 0/48 [00:00<?, ?it/s]

  [ML/DL Placeholder] Processing /content/drive/MyDrive/Collab datasets/public/videos/gallery_ded610c92b43.mp4
  [ML/DL Placeholder] Processing /content/drive/MyDrive/Collab datasets/public/videos/gallery_7ec0005149e9.mp4
  [ML/DL Placeholder] Processing /content/drive/MyDrive/Collab datasets/public/videos/gallery_bb43795a83cf.mp4
  [ML/DL Placeholder] Processing /content/drive/MyDrive/Collab datasets/public/videos/gallery_26378b4deb4a.mp4
  [ML/DL Placeholder] Processing /content/drive/MyDrive/Collab datasets/public/videos/gallery_d39773a55096.mp4
  [ML/DL Placeholder] Processing /content/drive/MyDrive/Collab datasets/public/videos/gallery_9bba1c63bffc.mp4
  [ML/DL Placeholder] Processing /content/drive/MyDrive/Collab datasets/public/videos/gallery_dc299067f8b7.mp4
  [ML/DL Placeholder] Processing /content/drive/MyDrive/Collab datasets/public/videos/gallery_c5cb6486e0e7.mp4
  [ML/DL Placeholder] Processing /content/drive/MyDrive/Collab datasets/public/videos/gallery_d4c6c46a1d89.mp4
 

Overview

Your objective is to watch each rendered 3D video and recover the final state of an object shuffle puzzle.

Each clip shows eight distinct PBR 3D objects on ten possible pedestal slots. The objects move through a two-stage shuffle while the camera moves and translucent occluders partially block the scene. Two final pedestal slots are empty decoys. The task is to output a structured JSON answer containing the final slot for every object and the empty final slots.

This is a video-to-structured-state tracking challenge, not a multiple-choice Q&A task. Strong solutions need object identity tracking, temporal reasoning, occlusion handling, and final-state reconstruction from visual evidence.

The current package contains 96 videos: 12 base 3D renders with 8 deterministic variants per base. The public split contains 48 training videos in train.csv and 48 test videos in test.csv. Variants include mirrored, lighting-shifted, and temporally reversed clips. Slot labels are defined in the observed video coordinate system, so mirrored and reversed clips have their own corrected answers.
Differentiation

This challenge is not a yes/no reasoning task or a single-object tracking task. Each submission must recover a complete structured state ledger: all eight object-to-slot assignments, the two empty decoy slots, and optionally evidence frames. Mirrored and temporally reversed variants change the correct labels, so a solution must reason in the observed video coordinate system rather than memorize canonical scene layouts. The private score also includes worst-group robustness over held-out camera direction, occlusion, and layout conditions.
Dataset Files

    train.csv

        48 training videos with ground-truth JSON answers.

        Columns: id, video, prompt, objects, slot_count, difficulty, answer_json

    test.csv

        48 test videos without answers.

        Columns: id, video, prompt, objects, slot_count, difficulty

    sample_submission.csv

        Submission template.

        Columns: id, answer_json

    videos/

        MP4 files referenced by train.csv and test.csv.

        Format: H.264 MP4, 768 x 432 pixels, 12 FPS, 96 frames.

Slot Convention

Slot indices are fixed:

    0 to 4: front row, left to right

    5 to 9: back row, left to right

Each video contains the same object ids:


helmet, avocado, boombox, lantern, camera, bottle, duck, toy_car

Submission Format

Submit a CSV with exactly these columns:


id,answer_json

gallery_012345abcdef_state,"{""final_slot_by_object"":{""helmet"":7,""avocado"":1,""boombox"":8,""lantern"":2,""camera"":4,""bottle"":0,""duck"":6,""toy_car"":3},""empty_slots"":[5,9],""evidence_frames"":{""helmet"":[83],""avocado"":[90]}}"

evidence_frames is optional but receives a small amount of credit when it contains valid frame indices from 0 to 95.
Evaluation

Rows are scored with partial credit:


row_score = 0.78 * object_slot_accuracy

          + 0.17 * empty_slot_jaccard

          + 0.05 * evidence_frame_validity

The final score also includes robustness terms for held-out split groups, OOD axes, and difficulty settings:


score = 0.70 * mean_row_score

      + 0.14 * worst_split_group_score

      + 0.10 * worst_ood_axis_score

      + 0.06 * worst_difficulty_score

Higher is better. A perfect submission scores 1.0.
Method Requirements

The intended solution space includes video transformers, object-centric trackers, multimodal sequence models, temporal memory systems, and world-model style state estimators.

Do not use:

    Hardcoded answers for specific test videos, ids, or filenames

    Manual lookup tables that map video names or ids directly to predictions

    Reverse-engineering the data generation process instead of solving from the video

    Non-public labels or metadata not included in the released public files

    Memorized test-set answers instead of model inference
